# chisel — HTTP Tunnel

chisel creates TCP tunnels over HTTP secured with SSH. No config file — only CLI flags.

## Tunnel mode reference

In [ ]:
modes = {
    "Forward tunnel": {
        "server": "chisel server --port 8080 --auth user:pass",
        "client": "chisel client --auth user:pass http://server:8080 <srv-port>:<local-host>:<local-port>",
        "example": "chisel client --auth user:pass http://server:8080 9000:localhost:3000",
        "effect": "traffic to server:9000 → client localhost:3000",
    },
    "Reverse tunnel": {
        "server": "chisel server --port 8080 --auth user:pass --reverse",
        "client": "chisel client --auth user:pass http://server:8080 R:<srv-port>:<target>:<port>",
        "example": "chisel client --auth user:pass http://server:8080 R:2222:localhost:22",
        "effect": "server:2222 → client localhost:22",
    },
    "Reverse SOCKS5": {
        "server": "chisel server --port 8080 --auth user:pass --reverse",
        "client": "chisel client --auth user:pass http://server:8080 R:1080:socks",
        "example": "curl --proxy socks5h://server:1080 https://internal-host",
        "effect": "server SOCKS5:1080 routes through client network",
    },
}

for mode, info in modes.items():
    print(f"=== {mode} ===")
    for k, v in info.items():
        print(f"  {k:<10}: {v}")
    print()

## Compose a chisel client command

In [ ]:
def chisel_client(server_url, auth, tunnels, keepalive="30s", retry=10):
    tunnel_args = " ".join(tunnels)
    return (f"chisel client "
            f"--auth {auth} "
            f"--keepalive {keepalive} "
            f"--max-retry-count {retry} "
            f"{server_url} "
            f"{tunnel_args}")

# Forward: expose local :3000 via server :9000
cmd1 = chisel_client("http://server.example.com:8080", "user:changeme",
                     ["9000:localhost:3000"])

# Reverse SOCKS5 + expose a second service
cmd2 = chisel_client("http://server.example.com:8080", "user:changeme",
                     ["R:1080:socks", "R:8888:localhost:3000"])

for cmd in (cmd1, cmd2):
    print(cmd)
    print()